# Executive Conflict Resolution — GRPO Training
Trains `Qwen/Qwen2.5-1.5B-Instruct` against the live OpenEnv using GRPO.
Set `HF_TOKEN` and `ENV_URL` in cell 2 before running.

In [ ]:
!pip install -q trl transformers accelerate peft requests datasets

In [ ]:
import os
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'
os.environ['ENV_URL'] = 'https://notAathi-OpenEvnHack.hf.space'
HF_TOKEN = os.environ['HF_TOKEN']
ENV_URL = os.environ['ENV_URL']

In [ ]:
import requests, json, uuid

def env_reset(task_id='hard'):
    sid = str(uuid.uuid4())
    r = requests.post(f'{ENV_URL}/reset', json={'session_id': sid, 'task_id': task_id})
    return r.json(), sid

def env_step(action, sid):
    return requests.post(f'{ENV_URL}/step', json={'session_id': sid, 'action': action}).json()

def env_score(sid):
    return requests.get(f'{ENV_URL}/score/{sid}').json().get('final_score', 0.0)

obs, sid = env_reset('hard')
print(f'ENV connected. Items: {len(obs["items"])}, first: {obs["items"][0]["title"]}')

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto', token=HF_TOKEN)
print('Model loaded:', MODEL_ID)

In [ ]:
import json

SYSTEM_PROMPT = """You are an expert executive assistant resolving workplace conflicts.
Respond with a JSON object only:
{"item_id": "c1", "conflict_type": "scheduling|deadline|delegation|social", "resolution": "reschedule|decline|delegate|accept|escalate", "message": "your message here"}"""

def make_prompt(item, context, instructions):
    return f"{context}\n\n{instructions}\n\nID: {item['id']}\nTitle: {item['title']}\nDescription: {item['description']}\nParticipants: {', '.join(item['participants'])}\nUrgency: {item['urgency']}"

def parse_action(raw, item_id):
    raw = raw.strip()
    s, e = raw.find('{'), raw.rfind('}') + 1
    if s == -1 or e == 0:
        return {'item_id': item_id, 'conflict_type': 'scheduling', 'resolution': 'reschedule', 'message': ''}
    try:
        d = json.loads(raw[s:e])
        return {'item_id': d.get('item_id', item_id), 'conflict_type': d.get('conflict_type', 'scheduling'),
                'resolution': d.get('resolution', 'reschedule'), 'message': d.get('message', '')}
    except:
        return {'item_id': item_id, 'conflict_type': 'scheduling', 'resolution': 'reschedule', 'message': ''}

def run_episode(task_id='hard'):
    obs, sid = env_reset(task_id)
    done = False
    while not done and obs.get('items'):
        item = obs['items'][0]
        msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': make_prompt(item, obs['context'], obs['instructions'])}]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=150, temperature=0.1, do_sample=True)
        raw = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        result = env_step(parse_action(raw, item['id']), sid)
        obs, done = result['observation'], result['done']
    return env_score(sid)

print('Running baseline (before training)...')
baseline_scores = [run_episode('hard') for _ in range(5)]
baseline_mean = sum(baseline_scores) / len(baseline_scores)
print(f'Baseline scores: {baseline_scores}')
print(f'Baseline mean:   {baseline_mean:.4f}')

In [ ]:
from datasets import Dataset

def collect_samples(n=100):
    samples = []
    while len(samples) < n:
        obs, sid = env_reset('hard')
        for item in obs.get('items', []):
            msgs = [{'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': make_prompt(item, obs['context'], obs['instructions'])}]
            text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
            samples.append({'prompt': text, 'item_id': item['id']})
    return Dataset.from_list(samples[:n])

dataset = collect_samples(100)
print(f'Dataset: {len(dataset)} samples')

In [ ]:
from trl import GRPOConfig, GRPOTrainer

reward_log = []

def reward_fn(completions, prompts=None, **kwargs):
    rewards = []
    item_ids = kwargs.get('item_id', ['c1'] * len(completions))
    for i, completion in enumerate(completions):
        try:
            obs, sid = env_reset('hard')
            item_id = item_ids[i] if i < len(item_ids) else 'c1'
            target = next((it for it in obs['items'] if it['id'] == item_id), obs['items'][0])
            action = parse_action(completion, target['id'])
            result = env_step(action, sid)
            reward = float(result['reward']['value'])
        except:
            reward = 0.01
        rewards.append(reward)
    reward_log.extend(rewards)
    return rewards

model.generation_config.max_new_tokens = 150

training_args = GRPOConfig(
    output_dir='/tmp/grpo-conflict',  # /tmp avoids disk space issues
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-6,
    logging_steps=5,
    save_strategy='no',              # no checkpoints = no disk crash
    report_to='none',
    bf16=True,
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=reward_fn,
)

print('Starting GRPO training...')
trainer.train()
print(f'Training done. Reward samples collected: {len(reward_log)}')

In [ ]:
print('Running post-training eval...')
after_scores = [run_episode('hard') for _ in range(5)]
after_mean = sum(after_scores) / len(after_scores)
print(f'Before: {baseline_scores} → {baseline_mean:.4f}')
print(f'After:  {after_scores} → {after_mean:.4f}')
print(f'Improvement: +{after_mean - baseline_mean:.4f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

window = 10
smoothed = np.convolve(reward_log, np.ones(window)/window, mode='valid') if len(reward_log) >= window else reward_log

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Reward curve
axes[0].plot(reward_log, alpha=0.3, color='steelblue', label='reward per step')
if len(reward_log) >= window:
    axes[0].plot(range(window-1, len(reward_log)), smoothed, color='steelblue', linewidth=2, label=f'{window}-step avg')
axes[0].set_xlabel('Training Step')
axes[0].set_ylabel('Reward')
axes[0].set_title('GRPO Reward Curve')
axes[0].legend()

# Before vs After
axes[1].bar(['Before', 'After'], [baseline_mean, after_mean], color=['#e74c3c', '#2ecc71'], width=0.4)
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Mean Final Score (Hard Task)')
axes[1].set_title(f'Before: {baseline_mean:.3f} → After: {after_mean:.3f}')
for i, v in enumerate([baseline_mean, after_mean]):
    axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.suptitle('Executive Conflict Resolution — GRPO Training Results', fontsize=13)
plt.tight_layout()
plt.savefig('training_summary.png', dpi=150)
plt.show()
print('Saved training_summary.png — download from file browser on the left')

In [ ]:
from huggingface_hub import login
login(token=HF_TOKEN)
model.push_to_hub('notAathi/conflict-resolution-grpo', token=HF_TOKEN)
tokenizer.push_to_hub('notAathi/conflict-resolution-grpo', token=HF_TOKEN)
print('Pushed to: https://huggingface.co/notAathi/conflict-resolution-grpo')